In [7]:
import torch
import torch.nn as nn
from torchvision import models

# RE-DÉFINITION DU DEVICE (au cas où le notebook l'a oublié)
device = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print("Device utilisé :", device)

# 1. Chargement de ResNet18 avec ses poids par défaut (pré-entraînés)
weights = models.ResNet18_Weights.DEFAULT
model_resnet = models.resnet18(weights=weights)

# 2. On fige les paramètres du modèle (Freeze)
for param in model_resnet.parameters():
    param.requires_grad = False

# 3. On remplace la dernière couche linéaire pour l'adapter à CIFAR-10
num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_ftrs, 10)

# 4. Envoi du modèle sur l'accélérateur (cuda/mps/cpu)
model_resnet = model_resnet.to(device)

# 5. Configuration de l'optimiseur (uniquement sur la couche fc)
optimizer = torch.optim.AdamW(model_resnet.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("ResNet18 prêt et configuré pour le Transfer Learning !")

Device utilisé : cuda
ResNet18 prêt et configuré pour le Transfer Learning !


In [8]:
# 1. Définition du Scheduler (divise le LR par 10 si la perte de test ne baisse pas pendant 2 époques)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.1)

# 2. Définition de la classe EarlyStopping
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0):
        self.patience = patience     # Nombre d'époques à attendre avant d'arrêter
        self.min_delta = min_delta   # Amélioration minimale requise
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            # Optionnel : sauvegarder les meilleurs poids ici si tu veux
        else:
            self.counter += 1
            print(f"--> EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=3)

In [9]:
# Assure-toi d'avoir installé torchmetrics avant : !pip install torchmetrics
import torchmetrics
from tqdm import tqdm

# Initialisation des métriques
metric_collection = torchmetrics.MetricCollection([
    torchmetrics.Accuracy(task="multiclass", num_classes=10),
    torchmetrics.Precision(task="multiclass", num_classes=10, average="macro"),
    torchmetrics.F1Score(task="multiclass", num_classes=10, average="macro")
]).to(device)

def train_loop_resnet(dataloader, model, loss_fn, optimizer, epoch, writer):
    model.train()
    progress_bar = tqdm(dataloader, desc=f"Époque {epoch} [Train]", leave=False)
    for batch, (X, y) in enumerate(progress_bar):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        global_step = epoch * len(dataloader) + batch
        writer.add_scalar("Loss/Train_ResNet", loss.item(), global_step)

def test_loop_resnet(dataloader, model, loss_fn, epoch, writer):
    model.eval()
    metric_collection.reset() # Reset des métriques à chaque début de test
    total_loss = 0
    
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            total_loss += loss_fn(pred, y).item()
            metric_collection.update(pred, y) # Envoi des prédictions à torchmetrics
            
    avg_loss = total_loss / len(dataloader)
    metrics = metric_collection.compute() # Calcul final
    
    # Logs TensorBoard
    writer.add_scalar("Loss/Test_ResNet", avg_loss, epoch)
    writer.add_scalar("Accuracy/Test_ResNet", metrics['MulticlassAccuracy'].item(), epoch)
    
    print(f"--> Époque {epoch} : Acc: {metrics['MulticlassAccuracy']:.4f} | F1: {metrics['MulticlassF1Score']:.4f} | Loss: {avg_loss:.4f}")
    return avg_loss

In [10]:
import torchvision
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter

# 1. On s'assure d'avoir TensorBoard prêt
writer = SummaryWriter("runs/resnet18_cifar10_experiment")

# 2. On recharge les transformations et les DataLoaders
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2023, 0.1994, 0.2010)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar10_mean, std=cifar10_std)
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar10_mean, std=cifar10_std)
])

dataset_train = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform_train, download=True)
dataset_test = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform_test, download=True)

dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=64, shuffle=True)
dataloader_test = torch.utils.data.DataLoader(dataset_test, batch_size=64, shuffle=False)

print("Dataloaders et TensorBoard initialisés avec succès !")

Dataloaders et TensorBoard initialisés avec succès !


In [11]:
def run_resnet_training():
    epochs = 10
    for t in range(epochs):
        print(f"\n=== Époque {t+1}/{epochs} ===")
        train_loop_resnet(dataloader_train, model_resnet, criterion, optimizer, t, writer)
        val_loss = test_loop_resnet(dataloader_test, model_resnet, criterion, t, writer)
        
        # Mise à jour du scheduler basé sur la perte de validation
        scheduler.step(val_loss)
        
        # Vérification du early stopping
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping déclenché ! Fin de l'entraînement.")
            break
            
    writer.close()

run_resnet_training()


=== Époque 1/10 ===


--> Époque 0 : Acc: 0.3777 | F1: 0.3698 | Loss: 1.8057

=== Époque 2/10 ===


--> Époque 1 : Acc: 0.4021 | F1: 0.3952 | Loss: 1.7528

=== Époque 3/10 ===


--> Époque 2 : Acc: 0.4062 | F1: 0.4020 | Loss: 1.7384

=== Époque 4/10 ===


--> Époque 3 : Acc: 0.3936 | F1: 0.3887 | Loss: 1.7565
--> EarlyStopping counter: 1/3

=== Époque 5/10 ===


--> Époque 4 : Acc: 0.4030 | F1: 0.4031 | Loss: 1.7366

=== Époque 6/10 ===


--> Époque 5 : Acc: 0.3975 | F1: 0.3963 | Loss: 1.7757
--> EarlyStopping counter: 1/3

=== Époque 7/10 ===


--> Époque 6 : Acc: 0.3930 | F1: 0.3834 | Loss: 1.7888
--> EarlyStopping counter: 2/3

=== Époque 8/10 ===


--> Époque 7 : Acc: 0.4049 | F1: 0.4019 | Loss: 1.7521
--> EarlyStopping counter: 3/3
Early stopping déclenché ! Fin de l'entraînement.
